# Skin Lesion Classification — 5-Class FAST Version (Structured)

A lighter, quicker version of the full 9-class pipeline: same three tables,
but restricted to **5 classes** and **fewer epochs** so a full run finishes
much faster (good for a first pass / sanity check before the full run).

This version keeps the exact same logic as the original notebook, but
organizes it into small, single-purpose classes/functions grouped by
responsibility (config, data, models, training, feature extraction,
efficiency benchmarking), so each stage is easier to read, test, and reuse.

1. **Table 1** — Transfer learning models (AlexNet → EfficientNet-B0)
2. **Table 2** — Classical classifiers on deep features
3. **Table 3** — Computational efficiency comparison

Dataset: https://www.kaggle.com/datasets/nodoubttome/skin-cancer9-classesisic
(download it yourself, unzip, upload to Drive — no Kaggle API needed here).

Run cells top to bottom. GPU runtime recommended (Runtime → Change runtime type → T4 GPU).

## 1. Install dependencies

In [1]:
!pip install -q thop xgboost


## 2. Dataset — mount Drive and set the path

Download the dataset yourself from Kaggle:
https://www.kaggle.com/datasets/nodoubttome/skin-cancer9-classesisic

Unzip it and upload the folder to your Google Drive (or upload the zip and
unzip it directly in Colab — see the commented option below). Then set
`DATA_ROOT` in the next cell to wherever the `Train`/`Test` subfolders end up.

In [5]:
from google.colab import drive
drive.mount('/content/drive')

import kagglehub
import os

# Download the dataset using kagglehub
# This will download and extract the dataset to a temporary directory
# and return the path to the extracted dataset.
dataset_path = kagglehub.dataset_download("nodoubttome/skin-cancer9-classesisic")

# Set DATA_ROOT to the path where the dataset was extracted.
# The output from the previous run indicates the main content is in a subfolder:
# Contents of DATA_ROOT: ['Skin cancer ISIC The International Skin Imaging Collaboration']
DATA_ROOT = os.path.join(dataset_path, "Skin cancer ISIC The International Skin Imaging Collaboration")

print("Contents of DATA_ROOT:", os.listdir(DATA_ROOT))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using Colab cache for faster access to the 'skin-cancer9-classesisic' dataset.
Contents of DATA_ROOT: ['Test', 'Train']


## 3. Config — a single dataclass holds every hyperparameter

Edit values here if needed; everything downstream reads from this one object instead of scattered globals.

In [6]:
import os
from dataclasses import dataclass, field
from typing import List

import torch


@dataclass
class Config:
    """All paths, hyperparameters, and model choices in one place."""

    data_root: str
    output_dir: str = "./results"

    # Chosen for a clinically meaningful mix of malignant + benign + decent sample counts
    class_subset: List[str] = field(default_factory=lambda: [
        "nevus",
        "melanoma",
        "basal cell carcinoma",
        "pigmented benign keratosis",
        "squamous cell carcinoma",
    ])

    val_fraction_of_train: float = 0.15
    random_seed: int = 42

    image_size: int = 224
    batch_size: int = 32
    feature_extract_epochs: int = 2   # reduced from 5 for speed
    fine_tune_epochs: int = 6         # reduced from 20 for speed
    learning_rate: float = 1e-4
    num_workers: int = 2
    num_timing_runs: int = 50

    cnn_models: List[str] = field(default_factory=lambda: [
        "alexnet", "vgg16", "vgg19", "resnet18",
        "resnet50", "resnet101", "densenet121", "efficientnet_b0",
    ])

    @property
    def train_dir(self) -> str:
        return os.path.join(self.data_root, "Train")

    @property
    def test_dir(self) -> str:
        return os.path.join(self.data_root, "Test")

    @property
    def device(self) -> torch.device:
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")

    def checkpoint_path(self, model_name: str) -> str:
        return os.path.join(self.output_dir, f"{model_name}_best.pt")

    def ensure_output_dir(self) -> None:
        os.makedirs(self.output_dir, exist_ok=True)


cfg = Config(data_root=DATA_ROOT)
cfg.ensure_output_dir()
DEVICE = cfg.device

print("Train classes found:", os.listdir(cfg.train_dir))
print("Test classes found:", os.listdir(cfg.test_dir))
print("Using device:", DEVICE)
print("Using class subset:", cfg.class_subset)


Train classes found: ['pigmented benign keratosis', 'melanoma', 'vascular lesion', 'actinic keratosis', 'squamous cell carcinoma', 'basal cell carcinoma', 'seborrheic keratosis', 'dermatofibroma', 'nevus']
Test classes found: ['pigmented benign keratosis', 'melanoma', 'vascular lesion', 'actinic keratosis', 'squamous cell carcinoma', 'basal cell carcinoma', 'seborrheic keratosis', 'dermatofibroma', 'nevus']
Using device: cuda
Using class subset: ['nevus', 'melanoma', 'basal cell carcinoma', 'pigmented benign keratosis', 'squamous cell carcinoma']


## 4. Dataset loading — filtered to the 5-class subset

Unlike the full pipeline (which uses every subfolder via `ImageFolder`), this
version only scans the 5 folders listed in `cfg.class_subset`.

Grouped into a single `SkinLesionData` class: scanning, splitting, and
`Dataset`/`DataLoader` construction are all methods on one object instead of
loose module-level functions.

In [7]:
import numpy as np
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from torchvision.datasets.folder import default_loader
import torchvision.transforms as T

IMG_EXTENSIONS = (".jpg", ".jpeg", ".png")


class FilteredImageDataset(Dataset):
    """Dataset over a pre-scanned (path, label) samples list, restricted to
    the configured class subset, with a configurable transform."""

    def __init__(self, samples, transform=None):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        image = default_loader(path)
        if self.transform:
            image = self.transform(image)
        return image, label


class SkinLesionData:
    """Scans the filtered class folders, splits train/val, and builds the
    three DataLoaders. All dataset-related logic lives here."""

    def __init__(self, cfg: Config):
        self.cfg = cfg
        self.classes: List[str] = []
        self.class_to_idx: dict = {}
        self.train_ds = self.val_ds = self.test_ds = None
        self.train_loader = self.val_loader = self.test_loader = None

    @staticmethod
    def _scan_filtered_samples(root, allowed_classes):
        """Like torchvision.datasets.ImageFolder, but only scans the given
        subfolders instead of every subfolder under root."""
        classes = sorted(allowed_classes)
        class_to_idx = {c: i for i, c in enumerate(classes)}
        samples = []
        for c in classes:
            class_dir = os.path.join(root, c)
            if not os.path.isdir(class_dir):
                raise FileNotFoundError(f"Expected class folder not found: {class_dir}")
            for fname in sorted(os.listdir(class_dir)):
                if fname.lower().endswith(IMG_EXTENSIONS):
                    samples.append((os.path.join(class_dir, fname), class_to_idx[c]))
        return samples, classes, class_to_idx

    @staticmethod
    def _get_transforms(image_size: int, train: bool):
        normalize = T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        if train:
            return T.Compose([
                T.Resize((image_size, image_size)),
                T.RandomHorizontalFlip(p=0.5),
                T.RandomVerticalFlip(p=0.5),
                T.RandomAffine(degrees=0, translate=(0.2, 0.2), shear=0.2),
                T.RandomResizedCrop(image_size, scale=(0.8, 1.0)),
                T.ToTensor(),
                normalize,
            ])
        return T.Compose([T.Resize((image_size, image_size)), T.ToTensor(), normalize])

    def load(self) -> "SkinLesionData":
        cfg = self.cfg
        train_samples_all, self.classes, self.class_to_idx = self._scan_filtered_samples(
            cfg.train_dir, cfg.class_subset
        )
        test_samples, _, _ = self._scan_filtered_samples(cfg.test_dir, cfg.class_subset)

        labels = np.array([lbl for _, lbl in train_samples_all])
        indices = np.arange(len(train_samples_all))
        train_idx, val_idx = train_test_split(
            indices, test_size=cfg.val_fraction_of_train, stratify=labels,
            random_state=cfg.random_seed,
        )
        train_samples = [train_samples_all[i] for i in train_idx]
        val_samples = [train_samples_all[i] for i in val_idx]

        self.train_ds = FilteredImageDataset(train_samples, self._get_transforms(cfg.image_size, train=True))
        self.val_ds = FilteredImageDataset(val_samples, self._get_transforms(cfg.image_size, train=False))
        self.test_ds = FilteredImageDataset(test_samples, self._get_transforms(cfg.image_size, train=False))

        self.train_loader = DataLoader(self.train_ds, batch_size=cfg.batch_size, shuffle=True, num_workers=cfg.num_workers)
        self.val_loader = DataLoader(self.val_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers)
        self.test_loader = DataLoader(self.test_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers)
        return self

    def unshuffled_train_loader(self) -> DataLoader:
        """Used by feature extraction, where sample order must be stable."""
        return DataLoader(self.train_ds, batch_size=self.cfg.batch_size, shuffle=False, num_workers=self.cfg.num_workers)


data = SkinLesionData(cfg).load()
classes = data.classes
print("Classes used (alphabetical order = label indices):", classes)
print(f"Train: {len(data.train_ds)}  Val: {len(data.val_ds)}  Test: {len(data.test_ds)}")


Classes used (alphabetical order = label indices): ['basal cell carcinoma', 'melanoma', 'nevus', 'pigmented benign keratosis', 'squamous cell carcinoma']
Train: 1541  Val: 273  Test: 80


## 5. Model builders (8 transfer-learning architectures)

Same architecture-swapping logic, wrapped in a `ModelFactory` so building a model and freezing/unfreezing its backbone are two clearly named class methods instead of free functions.

In [8]:
import torch.nn as nn
from torchvision import models


class ModelFactory:
    """Builds any of the 8 supported architectures with a replaced classifier
    head, and toggles backbone trainability for the two-stage fine-tuning."""

    @staticmethod
    def build(name: str, num_classes: int) -> nn.Module:
        if name == "alexnet":
            m = models.alexnet(weights="IMAGENET1K_V1")
            m.classifier[6] = nn.Linear(m.classifier[6].in_features, num_classes)
        elif name == "vgg16":
            m = models.vgg16(weights="IMAGENET1K_V1")
            m.classifier[6] = nn.Linear(m.classifier[6].in_features, num_classes)
        elif name == "vgg19":
            m = models.vgg19(weights="IMAGENET1K_V1")
            m.classifier[6] = nn.Linear(m.classifier[6].in_features, num_classes)
        elif name == "resnet18":
            m = models.resnet18(weights="IMAGENET1K_V1")
            m.fc = nn.Linear(m.fc.in_features, num_classes)
        elif name == "resnet50":
            m = models.resnet50(weights="IMAGENET1K_V2")
            m.fc = nn.Linear(m.fc.in_features, num_classes)
        elif name == "resnet101":
            m = models.resnet101(weights="IMAGENET1K_V2")
            m.fc = nn.Linear(m.fc.in_features, num_classes)
        elif name == "densenet121":
            m = models.densenet121(weights="IMAGENET1K_V1")
            m.classifier = nn.Linear(m.classifier.in_features, num_classes)
        elif name == "efficientnet_b0":
            m = models.efficientnet_b0(weights="IMAGENET1K_V1")
            m.classifier[1] = nn.Linear(m.classifier[1].in_features, num_classes)
        else:
            raise ValueError(f"Unknown model: {name}")
        return m

    @staticmethod
    def set_backbone_trainable(model: nn.Module, trainable: bool) -> None:
        for param in model.parameters():
            param.requires_grad = trainable
        last_layer = list(model.children())[-1]
        for param in last_layer.parameters():
            param.requires_grad = True

    @staticmethod
    def strip_classifier_head(model: nn.Module, name: str) -> nn.Module:
        """Replaces the final classifier layer with Identity so the model
        emits pooled deep features instead of class logits."""
        if name == "alexnet" or name.startswith("vgg"):
            model.classifier[6] = nn.Identity()
        elif name.startswith("resnet"):
            model.fc = nn.Identity()
        elif name == "densenet121":
            model.classifier = nn.Identity()
        elif name == "efficientnet_b0":
            model.classifier[1] = nn.Identity()
        return model


## 6. Training / evaluation loops

The warmup → fine-tune two-stage procedure, one epoch step, and metric computation are grouped into a `ModelTrainer` class bound to one `Config`/`SkinLesionData` pair, so `train_one_model` no longer depends on module-level globals.

In [9]:
import copy
import time
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score


class ModelTrainer:
    """Runs the two-stage (feature-extraction warmup + full fine-tune)
    training procedure for one architecture and reports test-set metrics."""

    def __init__(self, cfg: Config, data: SkinLesionData, device: torch.device):
        self.cfg = cfg
        self.data = data
        self.device = device

    def _run_epoch(self, model, loader, criterion, optimizer=None):
        is_train = optimizer is not None
        model.train() if is_train else model.eval()
        total_loss, total_correct, total_samples = 0.0, 0, 0
        with torch.set_grad_enabled(is_train):
            for images, labels in loader:
                images, labels = images.to(self.device), labels.to(self.device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                if is_train:
                    optimizer.zero_grad()
                    loss.backward()
                    optimizer.step()
                total_loss += loss.item() * images.size(0)
                total_correct += (outputs.argmax(1) == labels).sum().item()
                total_samples += images.size(0)
        return total_loss / total_samples, total_correct / total_samples

    @staticmethod
    def _compute_metrics(all_labels, all_preds, all_probs) -> dict:
        acc = accuracy_score(all_labels, all_preds)
        precision = precision_score(all_labels, all_preds, average="weighted", zero_division=0)
        recall = recall_score(all_labels, all_preds, average="weighted", zero_division=0)
        f1 = f1_score(all_labels, all_preds, average="weighted", zero_division=0)
        try:
            auc = roc_auc_score(all_labels, all_probs, multi_class="ovr", average="weighted")
        except ValueError:
            auc = float("nan")
        return {
            "Accuracy (%)": round(acc * 100, 2),
            "Precision (%)": round(precision * 100, 2),
            "Recall (%)": round(recall * 100, 2),
            "F1-Score (%)": round(f1 * 100, 2),
            "AUC (%)": round(auc * 100, 2),
        }

    def evaluate(self, model, loader) -> dict:
        model.eval()
        all_labels, all_preds, all_probs = [], [], []
        with torch.no_grad():
            for images, labels in loader:
                images = images.to(self.device)
                outputs = model(images)
                probs = torch.softmax(outputs, dim=1).cpu().numpy()
                all_labels.extend(labels.numpy())
                all_preds.extend(probs.argmax(axis=1))
                all_probs.extend(probs)
        return self._compute_metrics(np.array(all_labels), np.array(all_preds), np.array(all_probs))

    def train_one_model(self, name: str, num_classes: int):
        cfg = self.cfg
        print(f"\n=== Training {name} ===")
        model = ModelFactory.build(name, num_classes).to(self.device)
        criterion = nn.CrossEntropyLoss()

        # Stage 1: feature extraction - freeze backbone, train head only
        ModelFactory.set_backbone_trainable(model, trainable=False)
        optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=cfg.learning_rate)
        for epoch in range(cfg.feature_extract_epochs):
            train_loss, train_acc = self._run_epoch(model, self.data.train_loader, criterion, optimizer)
            val_loss, val_acc = self._run_epoch(model, self.data.val_loader, criterion, optimizer=None)
            print(f"  [warmup {epoch+1}/{cfg.feature_extract_epochs}] train_acc={train_acc:.3f} val_acc={val_acc:.3f}")

        # Stage 2: fine-tune the whole network
        ModelFactory.set_backbone_trainable(model, trainable=True)
        optimizer = torch.optim.Adam(model.parameters(), lr=cfg.learning_rate)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode="min", factor=0.1 ** 0.5, patience=7, min_lr=0.5e-6
        )
        best_val_acc, best_state = 0.0, copy.deepcopy(model.state_dict())
        for epoch in range(cfg.fine_tune_epochs):
            train_loss, train_acc = self._run_epoch(model, self.data.train_loader, criterion, optimizer)
            val_loss, val_acc = self._run_epoch(model, self.data.val_loader, criterion, optimizer=None)
            scheduler.step(val_loss)
            if val_acc > best_val_acc:
                best_val_acc, best_state = val_acc, copy.deepcopy(model.state_dict())
            print(f"  [finetune {epoch+1}/{cfg.fine_tune_epochs}] train_acc={train_acc:.3f} val_acc={val_acc:.3f}")

        model.load_state_dict(best_state)
        metrics = self.evaluate(model, self.data.test_loader)
        metrics["Model"] = name
        torch.save(model.state_dict(), cfg.checkpoint_path(name))
        return metrics, model


## 7. Table 1 — Transfer Learning Models

Trains all 8 architectures in sequence. With 5 classes and reduced epochs
(2 warmup + 6 fine-tune), this should run noticeably faster than the full
9-class/20-epoch version — expect several minutes per model on a T4, not tens of minutes.

In [10]:
trainer = ModelTrainer(cfg, data, DEVICE)

table1_results = []
for name in cfg.cnn_models:
    start = time.time()
    metrics, _ = trainer.train_one_model(name, len(classes))
    metrics["Train Time (min)"] = round((time.time() - start) / 60, 1)
    table1_results.append(metrics)
    pd.DataFrame(table1_results).to_csv(os.path.join(cfg.output_dir, "table1_transfer_learning_results.csv"), index=False)

table1_df = pd.DataFrame(table1_results)
table1_df



=== Training alexnet ===
Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /root/.cache/torch/hub/checkpoints/alexnet-owt-7be5be79.pth


100%|██████████| 233M/233M [00:01<00:00, 156MB/s]


  [warmup 1/2] train_acc=0.504 val_acc=0.619
  [warmup 2/2] train_acc=0.620 val_acc=0.612
  [finetune 1/6] train_acc=0.662 val_acc=0.652
  [finetune 2/6] train_acc=0.692 val_acc=0.685
  [finetune 3/6] train_acc=0.737 val_acc=0.641
  [finetune 4/6] train_acc=0.758 val_acc=0.729
  [finetune 5/6] train_acc=0.781 val_acc=0.700
  [finetune 6/6] train_acc=0.779 val_acc=0.685

=== Training vgg16 ===
Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:09<00:00, 60.3MB/s]


  [warmup 1/2] train_acc=0.432 val_acc=0.557
  [warmup 2/2] train_acc=0.562 val_acc=0.590
  [finetune 1/6] train_acc=0.449 val_acc=0.623
  [finetune 2/6] train_acc=0.620 val_acc=0.564
  [finetune 3/6] train_acc=0.658 val_acc=0.670
  [finetune 4/6] train_acc=0.718 val_acc=0.685
  [finetune 5/6] train_acc=0.731 val_acc=0.707
  [finetune 6/6] train_acc=0.758 val_acc=0.696

=== Training vgg19 ===
Downloading: "https://download.pytorch.org/models/vgg19-dcbb9e9d.pth" to /root/.cache/torch/hub/checkpoints/vgg19-dcbb9e9d.pth


...

100%|██████████| 30.8M/30.8M [00:00<00:00, 151MB/s]


  [warmup 1/2] train_acc=0.221 val_acc=0.278
  [warmup 2/2] train_acc=0.287 val_acc=0.337
  [finetune 1/6] train_acc=0.606 val_acc=0.711
  [finetune 2/6] train_acc=0.752 val_acc=0.703
  [finetune 3/6] train_acc=0.796 val_acc=0.744
  [finetune 4/6] train_acc=0.836 val_acc=0.762
  [finetune 5/6] train_acc=0.860 val_acc=0.755
  [finetune 6/6] train_acc=0.881 val_acc=0.718

=== Training efficientnet_b0 ===
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 178MB/s]


  [warmup 1/2] train_acc=0.266 val_acc=0.333
  [warmup 2/2] train_acc=0.391 val_acc=0.429
  [finetune 1/6] train_acc=0.561 val_acc=0.630
  [finetune 2/6] train_acc=0.694 val_acc=0.696
  [finetune 3/6] train_acc=0.769 val_acc=0.733
  [finetune 4/6] train_acc=0.803 val_acc=0.747
  [finetune 5/6] train_acc=0.820 val_acc=0.744
  [finetune 6/6] train_acc=0.860 val_acc=0.762


,Accuracy (%),Precision (%),Recall (%),F1-Score (%),AUC (%),Model,Train Time (min)
0,61.25,64.86,61.25,59.27,88.57,alexnet,4.3
1,55.00,43.73,55.00,43.46,93.24,vgg16,5.5
2,56.25,63.82,56.25,45.94,89.41,vgg19,5.7
3,62.50,61.82,62.50,58.87,88.32,resnet18,4.3
4,66.25,68.21,66.25,61.29,93.28,resnet50,4.5
5,66.25,75.90,66.25,62.03,92.25,resnet101,5.2
6,65.00,57.94,65.00,56.64,91.46,densenet121,4.7
7,61.25,63.13,61.25,55.88,91.48,efficientnet_b0,4.3


## 8. Table 2 — Classical Classifiers on Deep Features

Picks the best-performing CNN from Table 1, strips its classifier head, and feeds
the pooled feature vectors into 7 classical classifiers.

The feature extraction and classical-classifier evaluation are grouped into a
`ClassicalClassifierBenchmark` class so the "extract once, fit many classifiers"
workflow reads as a single unit.

In [11]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC, LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from xgboost import XGBClassifier


class ClassicalClassifierBenchmark:
    """Extracts deep features once from a trained CNN, then fits and scores a
    fixed set of classical classifiers on those features."""

    def __init__(self, cfg: Config, data: SkinLesionData, device: torch.device):
        self.cfg = cfg
        self.data = data
        self.device = device

    def _build_feature_extractor(self, model_name: str, num_classes: int) -> nn.Module:
        model = ModelFactory.build(model_name, num_classes)
        model.load_state_dict(torch.load(self.cfg.checkpoint_path(model_name), map_location=self.device))
        model = ModelFactory.strip_classifier_head(model, model_name).to(self.device)
        return model

    def _extract_features(self, loader, model: nn.Module):
        model.eval()
        feats, labels = [], []
        with torch.no_grad():
            for images, y in loader:
                images = images.to(self.device)
                feats.append(model(images).cpu().numpy())
                labels.append(y.numpy())
        return np.concatenate(feats), np.concatenate(labels)

    @staticmethod
    def _evaluate_classifier(clf, X_test, y_test) -> dict:
        preds = clf.predict(X_test)
        acc = accuracy_score(y_test, preds)
        precision = precision_score(y_test, preds, average="weighted", zero_division=0)
        recall = recall_score(y_test, preds, average="weighted", zero_division=0)
        f1 = f1_score(y_test, preds, average="weighted", zero_division=0)
        try:
            probs = clf.predict_proba(X_test)
            auc = roc_auc_score(y_test, probs, multi_class="ovr", average="weighted")
        except (AttributeError, ValueError):
            auc = float("nan")
        return {
            "Accuracy (%)": round(acc * 100, 2),
            "Precision (%)": round(precision * 100, 2),
            "Recall (%)": round(recall * 100, 2),
            "F1-Score (%)": round(f1 * 100, 2),
            "AUC (%)": round(auc * 100, 2),
        }

    @staticmethod
    def default_classifiers(random_seed: int) -> dict:
        return {
            "Logistic Regression": LogisticRegression(max_iter=2000, n_jobs=-1),
            "Decision Tree": DecisionTreeClassifier(random_state=random_seed),
            "Random Forest": RandomForestClassifier(n_estimators=300, random_state=random_seed, n_jobs=-1),
            "K-Nearest Neighbors (KNN)": KNeighborsClassifier(n_neighbors=5),
            "Linear SVM": CalibratedClassifierCV(LinearSVC(max_iter=5000)),
            "RBF-SVM": SVC(kernel="rbf", probability=True, random_state=random_seed),
            "XGBoost": XGBClassifier(n_estimators=300, eval_metric="mlogloss", random_state=random_seed),
        }

    def run(self, feature_extractor_model: str, num_classes: int) -> pd.DataFrame:
        feat_model = self._build_feature_extractor(feature_extractor_model, num_classes)

        print("Extracting deep features for train set...")
        X_train, y_train = self._extract_features(self.data.unshuffled_train_loader(), feat_model)
        print("Extracting deep features for test set...")
        X_test, y_test = self._extract_features(self.data.test_loader, feat_model)
        print("Feature vector size:", X_train.shape[1])

        results = []
        for cls_name, clf in self.default_classifiers(self.cfg.random_seed).items():
            print(f"\n=== Training {cls_name} on deep features ===")
            clf.fit(X_train, y_train)
            metrics = self._evaluate_classifier(clf, X_test, y_test)
            metrics["Feature Extractor"] = "Deep Features"
            metrics["Classifier"] = cls_name
            results.append(metrics)
            pd.DataFrame(results).to_csv(os.path.join(self.cfg.output_dir, "table2_classifier_results.csv"), index=False)

        return pd.DataFrame(results)


# Automatically pick the best model from Table 1; override manually if you prefer.
FEATURE_EXTRACTOR_MODEL = table1_df.sort_values("Accuracy (%)", ascending=False).iloc[0]["Model"]
print("Using feature extractor:", FEATURE_EXTRACTOR_MODEL)

classical_benchmark = ClassicalClassifierBenchmark(cfg, data, DEVICE)
table2_df = classical_benchmark.run(FEATURE_EXTRACTOR_MODEL, len(classes))
table2_df


Using feature extractor: resnet101
Extracting deep features for train set...
Extracting deep features for test set...
Feature vector size: 2048

=== Training Logistic Regression on deep features ===

=== Training Decision Tree on deep features ===

=== Training Random Forest on deep features ===

=== Training K-Nearest Neighbors (KNN) on deep features ===

=== Training Linear SVM on deep features ===

=== Training RBF-SVM on deep features ===

=== Training XGBoost on deep features ===


,Accuracy (%),Precision (%),Recall (%),F1-Score (%),AUC (%),Feature Extractor,Classifier
0,67.50,75.26,67.50,63.33,89.80,Deep Features,Logistic Regression
1,56.25,58.60,56.25,51.49,72.66,Deep Features,Decision Tree
2,63.75,78.78,63.75,57.66,93.07,Deep Features,Random Forest
3,63.75,70.11,63.75,57.80,87.00,Deep Features,K-Nearest Neighbors (KNN)
4,66.25,72.26,66.25,61.66,88.14,Deep Features,Linear SVM
5,65.00,75.27,65.00,61.02,92.56,Deep Features,RBF-SVM
6,63.75,76.97,63.75,59.32,90.74,Deep Features,XGBoost


## 9. Table 3 — Computational Efficiency Comparison

Parameters, model size, FLOPs, and inference time for all 8 architectures (accuracy pulled from Table 1).

Wrapped in an `EfficiencyBenchmark` class so "load checkpoint → measure size/FLOPs/latency → merge with Table 1 accuracy" is one method call per model.

In [12]:
from thop import profile


class EfficiencyBenchmark:
    """Measures parameter count, on-disk size, FLOPs, and inference latency
    for a trained checkpoint, and merges in its Table 1 accuracy."""

    def __init__(self, cfg: Config, device: torch.device, num_timing_runs: int = 50):
        self.cfg = cfg
        self.device = device
        self.num_timing_runs = num_timing_runs

    @staticmethod
    def _model_size_mb(checkpoint_path: str) -> float:
        return os.path.getsize(checkpoint_path) / (1024 ** 2)

    def _measure_inference_time_ms(self, model: nn.Module, input_size: int) -> float:
        model.eval()
        dummy = torch.randn(1, 3, input_size, input_size).to(self.device)
        with torch.no_grad():
            for _ in range(10):
                model(dummy)
        if self.device.type == "cuda":
            torch.cuda.synchronize()
        start = time.time()
        with torch.no_grad():
            for _ in range(self.num_timing_runs):
                model(dummy)
        if self.device.type == "cuda":
            torch.cuda.synchronize()
        return ((time.time() - start) / self.num_timing_runs) * 1000

    def benchmark(self, name: str, num_classes: int, table1_df: pd.DataFrame) -> dict:
        print(f"\n=== Benchmarking {name} ===")
        model = ModelFactory.build(name, num_classes).to(self.device)
        checkpoint_path = self.cfg.checkpoint_path(name)
        model.load_state_dict(torch.load(checkpoint_path, map_location=self.device))
        size_mb = round(self._model_size_mb(checkpoint_path), 2)

        dummy_input = torch.randn(1, 3, self.cfg.image_size, self.cfg.image_size).to(self.device)
        flops, params = profile(model, inputs=(dummy_input,), verbose=False)
        inference_ms = self._measure_inference_time_ms(model, self.cfg.image_size)

        accuracy_row = table1_df[table1_df["Model"] == name]
        accuracy = accuracy_row.iloc[0]["Accuracy (%)"] if not accuracy_row.empty else None

        return {
            "Model": name,
            "Parameters (M)": round(params / 1e6, 2),
            "Model Size (MB)": size_mb,
            "FLOPs (G)": round(flops / 1e9, 2),
            "Inference Time (ms)": round(inference_ms, 2),
            "Accuracy (%)": accuracy,
        }

    def run(self, model_names, num_classes: int, table1_df: pd.DataFrame) -> pd.DataFrame:
        results = []
        for name in model_names:
            results.append(self.benchmark(name, num_classes, table1_df))
            pd.DataFrame(results).to_csv(os.path.join(self.cfg.output_dir, "table3_efficiency_results.csv"), index=False)
        return pd.DataFrame(results)


efficiency_benchmark = EfficiencyBenchmark(cfg, DEVICE, num_timing_runs=cfg.num_timing_runs)
table3_df = efficiency_benchmark.run(cfg.cnn_models, len(classes), table1_df)
table3_df



=== Benchmarking alexnet ===

=== Benchmarking vgg16 ===

=== Benchmarking vgg19 ===

=== Benchmarking resnet18 ===

=== Benchmarking resnet50 ===

=== Benchmarking resnet101 ===

=== Benchmarking densenet121 ===

=== Benchmarking efficientnet_b0 ===


,Model,Parameters (M),Model Size (MB),FLOPs (G),Inference Time (ms),Accuracy (%)
0,alexnet,57.02,217.54,0.71,2.10,61.25
1,vgg16,134.28,512.25,15.47,9.94,55.00
2,vgg19,139.59,532.51,19.63,11.77,56.25
3,resnet18,11.18,42.72,1.82,3.66,62.50
4,resnet50,23.52,90.02,4.13,8.46,66.25
5,resnet101,42.51,162.77,7.86,13.24,66.25
6,densenet121,6.96,27.13,2.90,15.27,65.00
7,efficientnet_b0,4.01,15.60,0.41,8.19,61.25


## 10. Download the results

CSV files matching your three Word-doc tables are in `./results/`.

In [ ]:
from google.colab import files

for fname in ["table1_transfer_learning_results.csv", "table2_classifier_results.csv", "table3_efficiency_results.csv"]:
    path = os.path.join(cfg.output_dir, fname)
    if os.path.exists(path):
        files.download(path)
